In [4]:
import os
import copy
import pandas as pd

infile = '../../crc_analysis_12177/data/merged_hgt.csv'
db_idir = '../../HGT_demo_file/DB.VFDB.anno'
outdir = '.'
fr_size = 1000

df = pd.read_csv(infile, header=0, index_col=None)
# db = pd.read_csv(db_file, header=0, index_col=0, sep='\t')

df.rename(columns={'receptor':'recipient'}, inplace=True)

In [5]:
def overlap(range1, range2):
    if range1[0] > range2[1] or range1[1] < range2[0]:
        return False
    else:
        return True

# styp = recipient or donor
def search_event(scaffold, db, range):
    scaffold_df = db[db['Chr'] == scaffold]
    valid_idx = []
    for idx in scaffold_df.index:
        if overlap(range, [scaffold_df.loc[idx, 'Start'], scaffold_df.loc[idx, 'End']]):
            valid_idx.append(idx)
    valid_df = scaffold_df.loc[valid_idx, ]
    return valid_df
    

def search_row(idx, df, db_dir, fr_size):
    tmp = '{}.VFDB.tsv'
    row = df.loc[idx, ]
    # for recipient
    recipient = row['recipient']
    chrom = recipient.split('_')[0]
    ifile = os.path.join(db_dir, tmp.format(chrom))
    db = pd.read_csv(ifile, header=0, index_col=None, sep='\t')
    recipient_range = [max(0, row['insert_locus']-fr_size), row['insert_locus']+fr_size]
    recipient_df = search_event(recipient, db, recipient_range)
    # for donor
    donor = row['donor']
    range = [max(0, row['delete_start'] - fr_size), row['delete_end'] + fr_size]
    chrom = donor.split('_')[0]
    ifile = os.path.join(db_dir, tmp.format(chrom))
    db = pd.read_csv(ifile, header=0, index_col=None, sep='\t')
    donor_df = search_event(donor, db, range)
    return recipient_df, donor_df 



In [6]:
result_anno = pd.DataFrame(columns=['id', 'sample', 'recipient_VF_n', 'recipient_VF_category', 'recipient_VF_list', 'donor_VF_n', 'donor_VF_category', 'donor_VF_list', 'recipient', 'insert_locus', 'donor', 'delete_start', 'delete_end', 'reverse_flag'])
MGE_result = pd.DataFrame()
for idx in df.index:
    recipient_df, donor_df = search_row(idx, df, db_idir, fr_size)
    id = 'HGT_c{}'.format(idx+1)
    sample = df.loc[idx, 'sample']
    recipient_MGE_n = recipient_df.shape[0]
    recipient_MGE_category = ';'.join(recipient_df['Category'])
    recipient_MGE_list = ';'.join(recipient_df['Name'])
    if recipient_MGE_n == 0:
        recipient_MGE_list = 'NA'
        recipient_MGE_category = 'NA'
    donor_MGE_n = donor_df.shape[0]
    donor_MGE_category = ';'.join(recipient_df['Category'])
    donor_MGE_list = ';'.join(recipient_df['Name'])
    if donor_MGE_n == 0:
        donor_MGE_list = 'NA'
        donor_MGE_category = 'NA'
    recipient = df.loc[idx, 'recipient']
    insert_locus = df.loc[idx, 'insert_locus']
    donor = df.loc[idx, 'donor']
    delete_start = df.loc[idx, 'delete_start']
    delete_end = df.loc[idx, 'delete_end']
    reverse_flag = df.loc[idx, 'reverse_flag']
    result_anno.loc[len(result_anno), ] = [id, sample, recipient_MGE_n, recipient_MGE_category, recipient_MGE_list, donor_MGE_n, donor_MGE_category, donor_MGE_list, recipient, insert_locus, donor, delete_start, delete_end, reverse_flag]
    #result_anno.iloc[len(result_anno), ] = [id, sample, recipient_HGTC_n, recipient_HGTC_list, donor_HGTC_n, donor_HGTC_list, recipient, insert_locus, donor, delete_start, delete_end, reverse_flag]
    merge_df = pd.concat([recipient_df, donor_df], ignore_index=True)
    if MGE_result.empty:
        MGE_result = copy.deepcopy(merge_df)
    else:
        MGE_result = pd.concat([MGE_result, merge_df], ignore_index=True)
MGE_result.drop_duplicates(inplace=True)
MGE_result.to_csv(os.path.join(outdir, 'output.VF_annotation.VF.tsv'), index=False, sep='\t')
result_anno.to_csv(os.path.join(outdir, 'output.VF_annotation.annotated.tsv'), index=False, sep='\t')
